In [ ]:
samples = ["A1", "A2", "B2", "C2", "D1"]

In [ ]:
import spatialdata as sd
import plotnine as p9
import scvi
import scanpy as sc

In [ ]:
# loading zarr
sdata = sd.read_zarr("/Users/cydricgeyskens/Documents/code/phd/spatial-transcriptomics/scverse/hpc-scripts/intermediate_results/202601012.zarr")
sdata

In [ ]:
# loading the model
scvi_model = scvi.model.SCVI.load("intermediate_results/scvi_model_20260112.pt")

In [ ]:
sdata.tables["D1_transcriptomics_filter_scvi_clusters"].obs["cell_type"].value_counts()

In [ ]:
adatas = []
for s in samples:
    a = sdata.tables[f"{s}_transcriptomics_filter_scvi_clusters"].copy()
    a.obs["sample"] = s
    adatas.append(a)

adata_scvi = ad.concat(adatas, join="outer")  

## Differential Expression analysis

In [ ]:
adata_scvi.obs["cell_type"].value_counts()

In [ ]:
cell_type_1 = "Astrocytes Protoplasmic"
cell_idx1 = adata_scvi.obs["cell_type"] == cell_type_1
print(sum(cell_idx1), "cells of type", cell_type_1)

cell_type_2 = "DG Granule Cells"
cell_idx2 = adata_scvi.obs["cell_type"] == cell_type_2
print(sum(cell_idx2), "cells of type", cell_type_2)

In [ ]:
de_change = model_scvi.differential_expression(idx1=cell_idx1, idx2=cell_idx2, mode="change")
de_change.head(30)

In [ ]:
de_change_uniform = model_scvi.differential_expression(
    idx1=cell_idx1,  # we use the same cells as chosen before
    idx2=cell_idx2,
    weights="uniform",
    batch_correction=True,
    mode="change",
)
de_change_uniform.head(60)

In [ ]:
de_change_uniform["log10_pscore"] = np.log10(de_change_uniform["proba_not_de"])
de_change_uniform = de_change_uniform.join(adata.var, how="inner")
de_change_uniform.head(60)

In [ ]:
de_change_importance = model_scvi.differential_expression(
    idx1=cell_idx1,  # we use the same cells as chosen before
    idx2=cell_idx2,
    weights="importance",
    filter_outlier_cells=True,
    batch_correction=True,
    mode="change",
)

In [ ]:
de_change_importance["log10_pscore"] = np.log10(de_change_importance["proba_not_de"])
de_change_importance = de_change_importance.join(adata.var, how="inner")
de_change_importance.head(60)